In [13]:
import pandas as pd
import TextProcessing
import DataCollecting
import DataProcessing

import warnings
warnings.filterwarnings("ignore")

DATASET = "data.csv"

tp = TextProcessing.TextProcessor()

data = pd.read_csv(DATASET)

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# Sample DataFrame (replace with your own data)
# data = pd.read_csv('your_data.csv')

# Ensure 'Keywords' is a list — if it's a string, convert it
def safe_eval_keywords(x):
    if isinstance(x, str):
        try:
            return eval(x)
        except:
            return []
    return x

# Apply the safe_eval_keywords function to convert Keywords to lists
data['Keywords'] = data['Keywords'].apply(safe_eval_keywords)

# Clean the data by removing rows with empty 'Description' or 'Keywords'
data = data[data['Description'].notna() & data['Keywords'].apply(lambda x: len(x) > 0)]

# Feature Extraction:
# 1. TF-IDF vectorization for Descriptions
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
X_description = tfidf.fit_transform(data['Description']).toarray()

# 2. Binary encoding of Keywords (MultiLabelBinarizer)
mlb = MultiLabelBinarizer()
X_keywords = mlb.fit_transform(data['Keywords'])

# Combine both feature sets: Description + Keywords
X_combined = np.hstack([X_description, X_keywords])

# Encode Titles (the target variable)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['Title'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.2f}")

# Print the classification report using the same classes the model was trained on
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Predict on a new description + keywords
def predict_title(description, keywords):
    # Transform description using TF-IDF
    description_vector = tfidf.transform([description]).toarray()
    
    # Transform keywords using MultiLabelBinarizer
    keyword_vector = mlb.transform([keywords])
    
    # Combine both vectors
    combined_vector = np.hstack([description_vector, keyword_vector])
    
    # Predict the title
    predicted_index = model.predict(combined_vector)[0]
    return label_encoder.inverse_transform([predicted_index])[0]

# Example usage
new_description = "A young wizard discovers his magical powers and fights evil forces."
new_keywords = ['wizard', 'magic', 'evil', 'young']
predicted_title = predict_title(new_description, new_keywords)
print(f"Predicted Movie Title: {predicted_title}")


Model accuracy: 0.00


ValueError: Number of classes, 65, does not match size of target_names, 209. Try specifying the labels parameter

In [27]:
# Example
example_keywords = tp.extract_keywords("It is the year 1250 B.C  Paris the Trojan prince kidnaps Helen the queen of Sparta, the Greek War against Troy ensues")
predicted = predict_title_from_keywords(example_keywords)
print("Predicted title:", predicted)

ValueError: X has 4116 features, but MultinomialNB is expecting 8871 features as input.